# Two-Proportion Inference

This notebook uses a two-proportion z-test to compare the proportion of current alcohol use between male and female students.

Significance level: α = 0.05

## Hypothesis

Let $p_m$ be the proportion of male students who currently used alcohol.

Let $p_f$ be the proportion of female students who currently used alcohol.

The null hypothesis is:

$$H_0: p_m - p_f = 0$$

The alternative hypothesis is:

$$H_A: p_m - p_f \neq 0$$

This is a two-sided test because the research question asks whether the proportions are different between male and female students.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

project_dir = Path("..")

processed_data_path = project_dir / "data" / "processed" / "cycle3_cleaned_alcohol_gender.csv"
figures_dir = project_dir / "outputs" / "figures"
tables_dir = project_dir / "outputs" / "tables"

df = pd.read_csv(processed_data_path)

df.head()

In [ ]:
group_summary = df.groupby("sex_group").agg(
    sample_size=("current_alcohol_use", "count"),
    alcohol_users=("current_alcohol_use", "sum"),
    alcohol_use_proportion=("current_alcohol_use", "mean")
)

female_success = group_summary.loc["Female", "alcohol_users"]
female_n = group_summary.loc["Female", "sample_size"]
female_prop = group_summary.loc["Female", "alcohol_use_proportion"]

male_success = group_summary.loc["Male", "alcohol_users"]
male_n = group_summary.loc["Male", "sample_size"]
male_prop = group_summary.loc["Male", "alcohol_use_proportion"]

difference = male_prop - female_prop

group_summary

In [ ]:
count = np.array([male_success, female_success])
nobs = np.array([male_n, female_n])

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative="two-sided"
)

ci_low, ci_high = confint_proportions_2indep(
    count1=male_success,
    nobs1=male_n,
    count2=female_success,
    nobs2=female_n,
    method="wald"
)

print("Male proportion:", male_prop)
print("Female proportion:", female_prop)
print("Difference (Male - Female):", difference)
print("95% CI:", ci_low, "to", ci_high)
print("z statistic:", z_stat)
print("p-value:", p_value)

In [ ]:
inference_table = pd.DataFrame({
    "Statistic": [
        "Female sample size",
        "Female alcohol users",
        "Female alcohol use proportion",
        "Male sample size",
        "Male alcohol users",
        "Male alcohol use proportion",
        "Difference in proportions (Male - Female)",
        "95% CI lower bound",
        "95% CI upper bound",
        "z statistic",
        "p-value"
    ],
    "Value": [
        female_n,
        female_success,
        female_prop,
        male_n,
        male_success,
        male_prop,
        difference,
        ci_low,
        ci_high,
        z_stat,
        p_value
    ]
})

inference_table.to_csv(tables_dir / "inference_table_alcohol_by_gender.csv", index=False)

inference_table

In [ ]:
plt.figure(figsize=(6, 4))

plt.errorbar(
    x=[difference],
    y=["Male - Female"],
    xerr=[[difference - ci_low], [ci_high - difference]],
    fmt="o",
    capsize=5
)

plt.axvline(0, linestyle="--")

plt.xlabel("Difference in Proportions")
plt.title("95% CI for Difference in Current Alcohol Use")

plt.tight_layout()
plt.savefig(figures_dir / "ci_plot_difference_alcohol_by_gender.png", dpi=300)
plt.show()

## Assumptions

The response variable is binary because each student is classified as either currently using alcohol or not currently using alcohol.

The two groups are male and female students, and they are treated as independent groups.

The sample sizes are large enough for a two-proportion z-test because both groups have many observations and both success and failure counts are large.

Since the data come from an observational survey, the result can show an association between sex and current alcohol use, but it cannot prove causation.

In [ ]:
summary_dir = project_dir / "outputs" / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)

final_summary = f"""
# Cycle 3 Final Summary

## Research Question

Is the proportion of current alcohol use different between male and female students?

## Variables

- Group variable: WhatIsYourSex
- Response variable: CurrentAlcoholUse

## Method

Because the response variable is binary, a two-proportion z-test was used.

Welch's two-sample t-test was not used because this project compares two proportions, not two means.

## Key Results

- Female alcohol use proportion: {female_prop:.4f}
- Male alcohol use proportion: {male_prop:.4f}
- Difference in proportions (Male - Female): {difference:.4f}
- 95% confidence interval: ({ci_low:.4f}, {ci_high:.4f})
- z statistic: {z_stat:.4f}
- p-value: {p_value:.4f}

## Conclusion

At the 0.05 significance level, we fail to reject the null hypothesis.

There is not enough statistical evidence to conclude that the proportion of current alcohol use is different between male and female students.

Because the data come from an observational survey, the result should be interpreted as an association, not as a causal relationship.
"""

with open(summary_dir / "final_summary.md", "w", encoding="utf-8") as file:
    file.write(final_summary)

print("Final summary saved to:", summary_dir / "final_summary.md")

## Final Interpretation

The estimated proportion of current alcohol use was 0.4458 for female students and 0.4577 for male students.

The estimated difference in proportions was 0.0119, calculated as male proportion minus female proportion. This means that male students had about 1.19 percentage points higher current alcohol use than female students in this sample.

The 95% confidence interval for the difference was from -0.0054 to 0.0292.

The two-proportion z-test gave a z statistic of 1.3442 and a p-value of 0.1789.

At the 0.05 significance level, we fail to reject the null hypothesis.

In context, there is not enough statistical evidence to conclude that the proportion of current alcohol use is different between male and female students.

Because this dataset comes from an observational survey, this result should be interpreted as a group difference or association, not as a causal relationship.

## Practical Significance

Although the male alcohol use proportion was slightly higher than the female alcohol use proportion, the estimated difference was only 0.0119, or about 1.19 percentage points.

This suggests that the observed difference was small in practical size. Together with the p-value greater than 0.05 and the confidence interval including 0, the result does not provide strong evidence of a meaningful difference in current alcohol use between male and female students.